# 02 — Feature Engineering

This notebook demonstrates the leakage-safe feature pipeline:

1. The **booking-time whitelist** — every feature must be available at the booking horizon.
2. The **forbidden columns** audit — anything that materialises post-booking is dropped at the *first* pipeline stage so it cannot accidentally reach the model.
3. The **train-fold-only historical encoder** — rolling delay rates per route, carrier, origin, and aircraft tail.


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
np.seterr(all="ignore")

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "src").exists():
    sys.path.insert(0, str(ROOT.parent))

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)


In [2]:
from src.config import PROCESSED_DIR
from src.features.booking_time import BookingTimeFeatureBuilder, leakage_audit_table
from src.features.historical import HistoricalDelayRateEncoder
from src.data.ec261 import label_eligible_delay

df = pd.read_parquet(PROCESSED_DIR / "flights.parquet")
print(f"Rows: {len(df):,}")


Rows: 149,999


## The leakage-audit table

This is the canonical table that goes into the report.  It enumerates every BTS column we considered and explicitly states whether it is allowed at booking time or forbidden because it materialises later.

In [3]:
audit = leakage_audit_table()
print(audit.to_string(index=False))


                 column    status                                           reason
               DISTANCE   allowed                     available at booking horizon
       CRS_ELAPSED_TIME   allowed                     available at booking horizon
     AIRCRAFT_AGE_YEARS   allowed                     available at booking horizon
     WX_PRECIP_FCST_24H   allowed                     available at booking horizon
       WX_WIND_FCST_24H   allowed                     available at booking horizon
 WX_VISIBILITY_FCST_24H   allowed                     available at booking horizon
WX_CONVECTIVE_INDEX_24H   allowed                     available at booking horizon
   SCHED_TURNAROUND_MIN   allowed                     available at booking horizon
             LEG_OF_DAY   allowed                     available at booking horizon
 ORIG_HOURLY_DEPARTURES   allowed                     available at booking horizon
      OP_UNIQUE_CARRIER   allowed                     available at booking horizon
    

## Booking-time feature builder in action

In [4]:
builder = BookingTimeFeatureBuilder()
df_booking = builder.fit_transform(df.head(20_000).copy())
forbidden_present = [c for c in ["DEP_DELAY", "ARR_DELAY", "CARRIER_DELAY"] if c in df_booking.columns]
allowed_present = [c for c in ["HOUR", "DAYOFWEEK", "MONTH", "DISTANCE_TIER"] if c in df_booking.columns]
print("Forbidden columns surviving:", forbidden_present)
print("Allowed columns derived:    ", allowed_present)


Forbidden columns surviving: []
Allowed columns derived:     ['HOUR', 'DAYOFWEEK', 'MONTH', 'DISTANCE_TIER']


## Historical delay rates (train-fold-only)

The encoder is fit on training rows and then applied to validation/test rows.  Critical leakage guard: when computing the rolling rate for flight `f` on day `t`, only flights with date strictly before `t` contribute.  Same-day or future rows are excluded by `searchsorted` with `side='left'`.

In [5]:
years = df["FL_DATE"].dt.year
train_mask = years.isin([2018, 2019, 2020, 2021, 2022])
test_mask = years.isin([2023, 2024])

X_tr = df[train_mask].head(60_000).copy()
y_tr = label_eligible_delay(X_tr)
X_te = df[test_mask].head(20_000).copy()
y_te = label_eligible_delay(X_te)

route_enc = HistoricalDelayRateEncoder(
    key_cols=["ORIGIN", "DEST"], windows_days=(30, 90, 365), smoothing=100,
)
route_enc.fit(X_tr, y_tr)
rates_train = route_enc.transform(X_tr)
rates_test = route_enc.transform(X_te)

print("Train route-rate sample (first 5 rows):")
print(pd.DataFrame(rates_train[:5], columns=route_enc.get_feature_names_out()))
print("\nGlobal fallback rate (used for unseen routes):", round(route_enc.global_rate_, 4))


Train route-rate sample (first 5 rows):
Empty DataFrame
Columns: [ORIGIN_DEST_DELAY_RATE_30D, ORIGIN_DEST_DELAY_RATE_90D, ORIGIN_DEST_DELAY_RATE_365D]
Index: []

Global fallback rate (used for unseen routes): nan


## Sanity check: predictive correlation

A leaky historical encoder would correlate ~1.0 with the label on training data.  A well-implemented one correlates only modestly — the rate measures past behaviour, not the current flight's outcome.

In [6]:
import numpy as np
corr_train = np.corrcoef(rates_train[:, 1], y_tr.to_numpy())[0, 1]
corr_test = np.corrcoef(rates_test[:, 1], y_te.to_numpy())[0, 1]
print(f"Pearson r(route 90D rate, y) on train: {corr_train:.3f}")
print(f"Pearson r(route 90D rate, y) on test:  {corr_test:.3f}")
print("Both should be modest and positive (no >0.6 → no leakage).")


Pearson r(route 90D rate, y) on train: nan
Pearson r(route 90D rate, y) on test:  nan
Both should be modest and positive (no >0.6 → no leakage).
